<div style="display: flex; align-items: center;">
    <h1>Replacing WOFOST72 partitioning with a trainable neural network</h1>
    <img src="https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png" width="150" style="margin-left: 20px;">
</div>

This notebook shows how to replace one internal component of <code>Wofost72</code> with a user-defined neural network while keeping the rest of the crop model unchanged.

The example replaces <code>self.part</code>, the **partitioning** component, with a neural module that predicts the four partitioning factors from the crop development stage <code>DVS</code>.

## 1. Software requirements

Install the latest version of <code>diffwofost</code>. This notebook also uses <code>matplotlib</code> for plotting.

In [1]:
# install required packages
!pip install diffwofost matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import copy
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from diffwofost.ml_models.crop.partitioning import DVS_Partitioning_NN, PartitioningMLP, PartitioningNN
from diffwofost.physical_models.config import Configuration, ComputeConfig
from diffwofost.physical_models.crop.wofost72 import Wofost72
from diffwofost.physical_models.soil.classic_waterbalance import WaterbalancePP
from diffwofost.physical_models.utils import EngineTestHelper
from diffwofost.physical_models.utils import get_test_data
from diffwofost.physical_models.utils import prepare_engine_input

warnings.filterwarnings("ignore", message="To copy construct from a tensor.*")
ComputeConfig.set_device("cpu")
ComputeConfig.set_dtype(torch.float64)

SyntaxError: invalid syntax (engine.py, line 1)

## 2. Load a reference WOFOST72 dataset

We use one of the standard PCSE WOFOST72 potential production test cases. The reference <code>LAI</code> series from the YAML file is the optimization target.

The same dataset is also run once with the default rule-based partitioning module so we can compare the learned partitioning fractions against the original WOFOST behavior.

In [ ]:
filename = "test_potentialproduction_wofost72_03.yaml"
url = f"https://raw.githubusercontent.com/ajwdewit/pcse/refs/heads/master/tests/test_data/{filename}"

if not Path(filename).exists():
    urllib.request.urlretrieve(url, filename)

print(f"Using data file: {filename}")

In [ ]:
test_data = get_test_data(filename)

crop_model_params = [
    "SPAN",
    "TDWI",
    "TBASE",
    "PERDL",
    "RGRLAI",
    "KDIFTB",
    "SLATB",
    "TSUMEM",
    "TBASEM",
    "TEFFMX",
    "TSUM1",
    "TSUM2",
    "DLO",
    "DLC",
    "DVSI",
    "DVSEND",
    "DTSMTB",
    "AMAXTB",
    "EFFTB",
    "TMPFTB",
    "TMNFTB",
    "Q10",
    "RMR",
    "RML",
    "RMS",
    "RMO",
    "RFSETB",
    "CFET",
    "DEPNR",
    "IAIRDU",
    "IOX",
    "CRAIRC",
    "SM0",
    "SMW",
    "SMFCF",
    "RDI",
    "RRI",
    "RDMCR",
    "RDMSOL",
    "RDRRTB",
    "RDRSTB",
    "SSATB",
    "SPA",
    "FRTB",
    "FLTB",
    "FSTB",
    "FOTB",
    "CVL",
    "CVO",
    "CVR",
    "CVS",
]

(crop_model_params_provider, weather_data_provider, agro_management_inputs, external_states) = (
    prepare_engine_input(test_data, crop_model_params)
)

expected_lai = torch.tensor(
    [float(item["LAI"]) for item in test_data["ModelResults"]],
    dtype=ComputeConfig.get_dtype(),
    device=ComputeConfig.get_device(),
)

reference_config = Configuration(
    CROP=Wofost72,
    SOIL=WaterbalancePP,
    OUTPUT_VARS=["DVS", "LAI", "FR", "FL", "FS", "FO"],
)

print(f"Number of time steps: {expected_lai.numel()}")

## 3. Load reusable neural partitioning models

The package already provides reusable trainable partitioning models and a PCSE-compatible wrapper in [diffwofost.ml_models.crop.partitioning](../../src/diffwofost/ml_models/crop/partitioning.py).

In this notebook we compare two alternatives:

1. `PartitioningMLP`: an simple MLP model with a single hidden layer and a direct mapping from `DVS` to the four partitioning logits
2. `PartitioningNN`: a more refined model that first expands `DVS` into a few stage-aware features and then uses a shared nonlinear trunk with separate output heads

Both models still enforce physically valid partitioning outputs: `FR` is mapped through a sigmoid, and `FL`, `FS`, and `FO` are mapped through a softmax so they sum to 1.

In [ ]:
partition_mlp = PartitioningMLP(hidden_size=32)
partition_nn = PartitioningNN(hidden_size=32)

print(partition_mlp)
print()
print(partition_nn)

## 4. Configure `Wofost72` to use a neural partitioning module

`Configuration` is the object that tells the Engine which high-level model classes to build and which variables to store. In this notebook, the important fields are:

1. `CROP=Wofost72`: use the standard WOFOST72 crop model
2. `SOIL=WaterbalancePP`: keep the standard potential-production soil water balance
3. `OUTPUT_VARS=[...]`: request the variables that we want to inspect or optimize against
4. `CROP_KWARGS={...}`: pass extra keyword arguments from the Engine into the `Wofost72` constructor

The key new piece is `CROP_KWARGS`. When the Engine creates the crop model, it now forwards those keyword arguments to `Wofost72`. That means the crop model can stay `Wofost72`, while selected internal modules are swapped during initialization instead of requiring a subclass or monkey patch.

Inside `CROP_KWARGS`, `Wofost72` accepts a `component_overrides` dictionary. Each entry can override one internal component by name, for example `partitioning`, `phenology`, `assimilation`, or `leaf_dynamics`. An override can provide:

- a replacement simulation-object class through `class`
- an optional PyTorch model through `model`
- optional constructor arguments through `kwargs`

Here we override only `partitioning`, so all other WOFOST72 modules keep their default implementations. The Engine still instantiates a normal `Wofost72`, but `Wofost72` sees that the `partitioning` component has been overridden and builds `DVS_Partitioning_NN` with whichever neural model we pass at runtime.

That lets us compare an intentionally underpowered baseline (`PartitioningMLP`) against a more refined model (`PartitioningNN`) under the same crop-model configuration.

In [ ]:
hybrid_config = Configuration(
    CROP=Wofost72,
    CROP_KWARGS={
        "component_overrides": {
            "partitioning": {
                "class": DVS_Partitioning_NN,
                "model": partition_nn,
            }
        }
    },
    SOIL=WaterbalancePP,
    OUTPUT_VARS=["DVS", "LAI", "FR", "FL", "FS", "FO"],
)

Now we run the hybrid crop model with a normal Engine configuration. The optimizer still works directly on the neural-network parameters, while `Wofost72` instantiates `DVS_Partitioning_NN` from the `component_overrides` entry in `CROP_KWARGS`.

The helper below is model-agnostic, so we can reuse the same simulation path for both `PartitioningNN` and `PartitioningMLP`.

In [ ]:
def run_hybrid_wofost72(partition_model):
    hybrid_config.CROP_KWARGS["component_overrides"]["partitioning"]["model"] = partition_model

    engine = EngineTestHelper(
        copy.deepcopy(crop_model_params_provider),
        weather_data_provider,
        agro_management_inputs,
        hybrid_config,
        external_states,
    )
    engine.run_till_terminate()
    results = engine.get_output()

    return {
        "DVS": torch.stack([item["DVS"] for item in results]),
        "LAI": torch.stack([item["LAI"] for item in results]),
        "FR": torch.stack([item["FR"] for item in results]),
        "FL": torch.stack([item["FL"] for item in results]),
        "FS": torch.stack([item["FS"] for item in results]),
        "FO": torch.stack([item["FO"] for item in results]),
    }

Before training, we can compare the default WOFOST72 run against both untrained neural alternatives. This gives a baseline for how much the intentionally simple linear model and the refined model already resemble the original partitioning behavior before optimization.

In [ ]:

def run_reference_wofost72():
    engine = EngineTestHelper(
        copy.deepcopy(crop_model_params_provider),
        weather_data_provider,
        agro_management_inputs,
        reference_config,
        external_states,
    )
    engine.run_till_terminate()
    results = engine.get_output()
    return {
        "DVS": torch.stack([item["DVS"] for item in results]),
        "LAI": torch.stack([item["LAI"] for item in results]),
        "FR": torch.stack([item["FR"] for item in results]),
        "FL": torch.stack([item["FL"] for item in results]),
        "FS": torch.stack([item["FS"] for item in results]),
        "FO": torch.stack([item["FO"] for item in results]),
    }

In [ ]:
reference_results = run_reference_wofost72()

initial_runs = {
    "PartitioningNN": run_hybrid_wofost72(partition_nn),
    "PartitioningMLP": run_hybrid_wofost72(partition_mlp),
}

for model_name, model_results in initial_runs.items():
    initial_loss = torch.mean(torch.abs(model_results["LAI"] - expected_lai))
    print(f"{model_name} initial LAI MAE: {initial_loss.detach().item():.4f}")
    print(f"{model_name} initial FR sample: {model_results['FR'][10].detach().item():.4f}")

## 5. Optimize both neural partitioning blocks

The optimizer sees only the parameters of the active neural model, but gradients are computed through the complete hybrid crop model executed by the helper function above.

The training objective is a weighted relative MAE on `LAI`: time steps with larger reference receive more weight, and the error is normalized by the weighted magnitude of the target series.

To keep the learned partitioning behavior better behaved, the notebook also adds two explicit regularization terms: a small L2 penalty on the neural-network weights and a temporal smoothness penalty on the predicted partitioning fractions.

The loop also uses adaptive learning-rate reduction, gradient clipping, and early stopping with best-weight restoration.

Because the helper is model-agnostic, we can train `PartitioningNN` and `PartitioningMLP` with the same procedure and compare the resulting trajectories.

In [ ]:
peak_lai = torch.max(expected_lai).clamp_min(1e-6)
loss_weights = 0.1 + expected_lai / peak_lai
loss_denom = torch.mean(loss_weights * torch.abs(expected_lai)).clamp_min(1e-6)

l2_weight = 1e-4
smoothness_weight = 5e-3
partition_vars = ("FR", "FL", "FS", "FO")


def compute_partition_regularization(model, predictions):
    parameter_penalty = torch.zeros(
        (), dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device()
    )
    for parameter in model.parameters():
        parameter_penalty = parameter_penalty + parameter.pow(2).mean()

    smoothness_penalty = torch.zeros(
        (), dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device()
    )
    for varname in partition_vars:
        values = predictions[varname]
        if values.numel() > 1:
            smoothness_penalty = smoothness_penalty + torch.mean(
                torch.diff(values, dim=0).pow(2)
            )

    return parameter_penalty, smoothness_penalty


def train_partition_model(
    model,
    model_name,
    lr=0.003,
    max_steps=60,
    patience=12,
    min_delta=1e-4,
    l2_weight=l2_weight,
    smoothness_weight=smoothness_weight,
    optimizer_weight_decay=0.0,
):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=optimizer_weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=8, min_lr=1e-4
    )

    best_loss = float("inf")
    best_step = -1
    best_state = copy.deepcopy(model.state_dict())
    loss_history = []
    lai_loss_history = []

    for step in range(max_steps):
        optimizer.zero_grad()
        predictions = run_hybrid_wofost72(model)
        lai_loss = torch.mean(
            loss_weights * torch.abs(predictions["LAI"] - expected_lai)
        ) / loss_denom
        parameter_penalty, smoothness_penalty = compute_partition_regularization(
            model, predictions
        )
        regularization_loss = (
            l2_weight * parameter_penalty + smoothness_weight * smoothness_penalty
        )
        total_loss = lai_loss + regularization_loss

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step(total_loss.detach())

        total_loss_value = total_loss.detach().cpu().item()
        lai_loss_value = lai_loss.detach().cpu().item()
        reg_loss_value = regularization_loss.detach().cpu().item()
        loss_history.append(total_loss_value)
        lai_loss_history.append(lai_loss_value)

        if total_loss_value < best_loss - min_delta:
            best_loss = total_loss_value
            best_step = step
            best_state = copy.deepcopy(model.state_dict())

        if step % 5 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(
                f"{model_name} | Step {step:03d} | LAI = {lai_loss_value:.4f} | Reg = {reg_loss_value:.4f} | Total = {total_loss_value:.4f} | LR = {current_lr:.5f}"
            )

        if step - best_step >= patience:
            print(f"{model_name} | Early stopping at step {step:03d}")
            break

    model.load_state_dict(best_state)
    trained_results = run_hybrid_wofost72(model)
    trained_loss = torch.mean(torch.abs(trained_results["LAI"] - expected_lai))
    trained_weighted_loss = (
        torch.mean(loss_weights * torch.abs(trained_results["LAI"] - expected_lai))
        / loss_denom
    )
    trained_parameter_penalty, trained_smoothness_penalty = (
        compute_partition_regularization(model, trained_results)
    )
    trained_regularization = (
        l2_weight * trained_parameter_penalty
        + smoothness_weight * trained_smoothness_penalty
    )

    return {
        "model": model,
        "loss_history": loss_history,
        "lai_loss_history": lai_loss_history,
        "results": trained_results,
        "best_objective": best_loss,
        "final_weighted_loss": trained_weighted_loss.detach().item(),
        "final_regularization": trained_regularization.detach().item(),
        "final_lai_mae": trained_loss.detach().item(),
    }


training_runs = {
    "PartitioningNN": train_partition_model(partition_nn, "PartitioningNN"),
    "PartitioningMLP": train_partition_model(partition_mlp, "PartitioningMLP"),
}

for model_name, run_info in training_runs.items():
    print(f"{model_name} best regularized objective: {run_info['best_objective']:.4f}")
    print(f"{model_name} final weighted relative LAI loss: {run_info['final_weighted_loss']:.4f}")
    print(f"{model_name} final regularization loss: {run_info['final_regularization']:.4f}")
    print(f"{model_name} final LAI MAE: {run_info['final_lai_mae']:.4f}")

Let's plot the trained LAI trajectories and the regularized optimization histories for both neural alternatives.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(expected_lai.detach().cpu().numpy(), label="Reference LAI", linewidth=2)
for model_name, run_info in training_runs.items():
    axes[0].plot(
        run_info["results"]["LAI"].detach().cpu().numpy(),
        label=f"{model_name} LAI",
        linewidth=2,
        linestyle="--",
    )
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("LAI")
axes[0].set_title("Reference vs. trained hybrid-model LAI")
axes[0].grid(alpha=0.3)
axes[0].legend()

for model_name, run_info in training_runs.items():
    axes[1].plot(run_info["loss_history"], linewidth=2, label=model_name)
axes[1].set_xlabel("Optimization step")
axes[1].set_ylabel("Regularized objective")
axes[1].set_title("Regularized training loss")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Inspect the learned partitioning curves

The next plot compares the two neural partitioning fractions with the default WOFOST72 partitioning module.

This is not the training target in this notebook. It is only a diagnostic that helps you see how each neural replacement differs from the original process representation while still fitting the observed <code>LAI</code>.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
curves = [
    ("FR", "Roots"),
    ("FL", "Leaves"),
    ("FS", "Stems"),
    ("FO", "Storage organs"),
]

for ax, (varname, title) in zip(axes.ravel(), curves, strict=False):
    ax.plot(
        reference_results["DVS"].detach().cpu().numpy(),
        reference_results[varname].detach().cpu().numpy(),
        label="Default WOFOST72",
        linewidth=2,
    )

    for model_name, run_info in training_runs.items():
        ax.plot(
            run_info["results"]["DVS"].detach().cpu().numpy(),
            run_info["results"][varname].detach().cpu().numpy(),
            label=model_name,
            linewidth=2,
            linestyle="--",
        )

    ax.set_title(title)
    ax.set_xlabel("DVS")
    ax.set_ylabel(varname)
    ax.grid(alpha=0.3)

axes[0, 0].legend()
plt.tight_layout()
plt.show()

You can use this comparison to see how much the final behavior depends on the neural architecture. In practice, `PartitioningNN` and `PartitioningMLP` can reach similar LAI fits while producing noticeably different partitioning trajectories, which is exactly why it is useful to compare both the training target and the internal diagnostic variables.

## 7. Recap

You can reuse the same pattern shown here for other internal WOFOST components, not only partitioning.

The general recipe is:

1. identify the module you want to replace and the variables it reads from the kiosk
2. load or define a PyTorch model that maps those inputs to the outputs that module is supposed to provide
3. plug that model into a compatible simulation-object wrapper
4. pass the replacement component through the crop-model configuration so `Wofost72` instantiates it during setup
5. choose a training target, run the hybrid model end to end, and optimize only the neural-network parameters

This notebook also shows that you can compare multiple NN architectures under exactly the same WOFOST configuration. That makes it easier to separate two questions: how well a model fits the target trajectory, and how closely its learned internal behavior still resembles the original process-based module.

The main thing to preserve is compatibility with the rest of the simulation: if your replacement publishes the right variables and follows the same calling pattern, you can usually test the effect of a neural alternative without rewriting the rest of the crop model.